# NLP Lab 01: Introduction to Natural Language Processing

## Setting Up NLP Libraries

We will use three libraries throughout this lab:

| Library | Best for | Language focus |
|---|---|---|
| **NLTK** (Natural Language Toolkit) | Teaching/classic NLP: tokenization, POS tagging, simple parsing, corpora | Mainly English |
| **spaCy** | Fast, production-grade pipelines: tokenization, POS, dependency parsing, NER | Many languages (via models), mainly English here |
| **PyThaiNLP** | Thai-specific NLP: word segmentation, POS tagging, NER, stopwords, transliteration | Thai |

Run the cell below once to install/download everything you need. If you're on a shared or offline machine, some downloads (like the spaCy model) may need to be run separately or may already be cached.

In [1]:
# Run this once. Uncomment if the packages are not already installed in your environment.
# %pip install -q nltk spacy pythainlp scikit-learn
# !python -m spacy download en_core_web_sm
print("If needed, uncomment the install lines above and re-run this cell.")

If needed, uncomment the install lines above and re-run this cell.


In [2]:
# import nltk

# # Download the small set of NLTK data packages we need for this lab.
# for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
#             "stopwords", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
#     try:
#         nltk.download(pkg, quiet=True)
#     except Exception as e:
#         print(f"Skipped {pkg}: {e}")

# print("NLTK data ready.")

## NLTK example (English)

NLTK gives us the classic NLP pipeline building blocks: tokenization, POS tagging, and a simple chunk-based NER.
- https://www.nltk.org//
- https://www.geeksforgeeks.org/nlp/nltk-tutorial/ 


In [3]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import stopwords

sentence = "Apple is looking at buying a U.K. startup for $1 billion."

# 1. Tokenization: split raw text into words/punctuation tokens
tokens = word_tokenize(sentence)
print("Tokens:", tokens)

# 2. Part-of-speech (POS) tagging: label each token's grammatical role
tagged = pos_tag(tokens)
print("\nPOS tags:", tagged)

# 3. Named Entity Recognition (a simple, rule/statistics-based chunker)
tree = ne_chunk(tagged)
print("\nNamed entities:")
for subtree in tree:
    if hasattr(subtree, "label"):
        entity_text = " ".join(word for word, tag in subtree.leaves())
        print(f"  {entity_text}  ->  {subtree.label()}")

# 4. Stopword removal (common, low-information words)
stop_words = set(stopwords.words("english"))
filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]
print("\nTokens without stopwords:", filtered)

Tokens: ['Apple', 'is', 'looking', 'at', 'buying', 'a', 'U.K.', 'startup', 'for', '$', '1', 'billion', '.']

POS tags: [('Apple', 'NNP'), ('is', 'VBZ'), ('looking', 'VBG'), ('at', 'IN'), ('buying', 'VBG'), ('a', 'DT'), ('U.K.', 'NNP'), ('startup', 'NN'), ('for', 'IN'), ('$', '$'), ('1', 'CD'), ('billion', 'CD'), ('.', '.')]

Named entities:
  Apple  ->  GPE

Tokens without stopwords: ['Apple', 'looking', 'buying', 'startup', 'billion']


## spaCy example (English)

spaCy bundles tokenization, POS tagging, dependency parsing, and NER into a single, fast pipeline (`nlp(text)`).
- https://spacy.io/
- https://www.tutorialspoint.com/spacy/index.htm

In [4]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise RuntimeError(
        "Model not found. Run: python -m spacy download en_core_web_sm"
    )

doc = nlp("Apple is looking at buying a U.K. startup for $1 billion.")

# 1. Linguistic Analysis (Tokens, POS, Dependency Parse)
print(f"{'TOKEN':<14} {'POS':<10} {'DEP':<14} {'HEAD':<14}")
print("-" * 52)
for token in doc:
    print(f"{token.text:<14} {token.pos_:<10} {token.dep_:<14} {token.head.text:<14}")

# 2. Named Entity Recognition (NER)
print("\n" + "=" * 65)
print(f"{'NAMED ENTITY':<20} {'LABEL':<10} {'DESCRIPTION'}")
print("-" * 65)
for ent in doc.ents:
    description = spacy.explain(ent.label_) or ""
    print(f"{ent.text:<20} {ent.label_:<10} {description}")
print("=" * 65)


TOKEN          POS        DEP            HEAD          
----------------------------------------------------
Apple          PROPN      nsubj          looking       
is             AUX        aux            looking       
looking        VERB       ROOT           looking       
at             ADP        prep           looking       
buying         VERB       pcomp          at            
a              DET        det            U.K.          
U.K.           PROPN      dobj           buying        
startup        NOUN       advcl          looking       
for            ADP        prep           startup       
$              SYM        quantmod       billion       
1              NUM        compound       billion       
billion        NUM        pobj           for           
.              PUNCT      punct          looking       

NAMED ENTITY         LABEL      DESCRIPTION
-----------------------------------------------------------------
Apple                ORG        Companies, agencies,

## PyThaiNLP example (Thai)

PyThaiNLP handles the Thai-specific problem we introduced in Section 1: **there are no spaces between Thai words**, so tokenization requires a dedicated algorithm/model rather than simply splitting on whitespace.
- https://pythainlp.org/

In [5]:
from pythainlp.tokenize import word_tokenize as th_word_tokenize
from pythainlp.tag import pos_tag as th_pos_tag
from pythainlp.corpus import thai_stopwords

thai_sentence = "ฉันชอบกินข้าวผัดกะเพราที่ร้านนี้มากเลยครับ"

# 1. Word tokenization (default engine: "newmm", a dictionary-based maximal matching + TCC)
th_tokens = th_word_tokenize(thai_sentence)
print("Thai tokens:", th_tokens)

# 2. POS tagging
th_tagged = th_pos_tag(th_tokens)
print("\nThai POS tags:", th_tagged)

# 3. Stopword removal
th_stopwords = thai_stopwords()
th_filtered = [w for w in th_tokens if w not in th_stopwords and w.strip() != ""]
print("\nThai tokens without stopwords:", th_filtered)

Thai tokens: ['ฉัน', 'ชอบ', 'กินข้าว', 'ผัดกะเพรา', 'ที่', 'ร้าน', 'นี้', 'มาก', 'เลย', 'ครับ']

Thai POS tags: [('ฉัน', 'PPRS'), ('ชอบ', 'VACT'), ('กินข้าว', 'NCMN'), ('ผัดกะเพรา', 'NCMN'), ('ที่', 'PREL'), ('ร้าน', 'NCMN'), ('นี้', 'DDAC'), ('มาก', 'ADVN'), ('เลย', 'ADVN'), ('ครับ', 'ADVN')]

Thai tokens without stopwords: ['ชอบ', 'กินข้าว', 'ผัดกะเพรา', 'ร้าน']


## Core Challenge Demo: Thai Word Segmentation Ambiguity

Let's revisit the ambiguous sentence from Section 1 and see how different tokenization **engines** inside PyThaiNLP can actually produce different segmentations — this is a live demonstration of why word segmentation is treated as its own hard NLP research problem for Thai, rather than a "solved" preprocessing step.

In [6]:
from pythainlp.tokenize import word_tokenize

sentences = [
    "อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน",
    "เขาตัดทรงผมใหม่แล้วดูดีมาก",
    
]

for sent in sentences:
    print(f"ประโยค: {sent}")
    for engine in ["newmm", "longest", "mm"]:
        try:
            result = word_tokenize(sent, engine=engine)
            print(f"  {engine:>10}: {result}")
        except Exception as e:
            print(f"  {engine:>10}: Error ({e})")
    print("-" * 40)


ประโยค: อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน
       newmm: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
     longest: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
          mm: ['อาจารย์', 'สอนภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
----------------------------------------
ประโยค: เขาตัดทรงผมใหม่แล้วดูดีมาก
       newmm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดู', 'ดีมาก']
     longest: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดี', 'มาก']
          mm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดีมาก']
----------------------------------------


## NLP Applications

### Text Classification

**Text classification** assigns a predefined label to a piece of text — e.g., sentiment (positive/negative), topic, spam/not-spam, or intent. It is a classic **NLU** task: text goes in, a structured label comes out.

Below is a minimal pipeline: turn text into numeric features (bag-of-words), then train a simple classifier. In real systems, the "text → numeric features" step is often replaced by upstream-learned embeddings (Word2Vec, BERT), but the classification idea stays the same.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Toy training data: (review, label) where 1 = positive, 0 = negative
train_texts = [
    "I love this movie, it was fantastic",
    "What a wonderful and amazing film",
    "This was a terrible and boring movie",
    "I hated every minute of this film",
    "Absolutely brilliant, I really enjoyed it",
    "Awful acting, a complete waste of time",
]
train_labels = [1, 1, 0, 0, 1, 0]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)

clf = LogisticRegression()
clf.fit(X_train, train_labels)

test_texts = ["This film was absolutely wonderful", "A boring and terrible experience"]
X_test = vectorizer.transform(test_texts)
predictions = clf.predict(X_test)

for text, pred in zip(test_texts, predictions):
    print(f"{text!r:55} -> {'positive' if pred == 1 else 'negative'}")

'This film was absolutely wonderful'                    -> positive
'A boring and terrible experience'                      -> negative


In [8]:
# Same idea, but for Thai text: we swap the tokenizer used by CountVectorizer
# for PyThaiNLP's word_tokenize, since Thai has no whitespace to split on.
from pythainlp.tokenize import word_tokenize as th_tokenize

th_train_texts = [
    "อาหารร้านนี้อร่อยมากแนะนำเลย",       # this restaurant's food is very delicious, recommended
    "บริการดีมากประทับใจสุดๆ",           # the service was great, very impressed
    "อาหารรสชาติแย่มากไม่อร่อยเลย",       # the food tasted very bad, not delicious at all
    "บริการห่วยแตกรอนานมากไม่ประทับใจ",   # terrible service, waited very long, not impressed
]
th_train_labels = [1, 1, 0, 0]

th_vectorizer = CountVectorizer(tokenizer=th_tokenize, token_pattern=None)
X_th_train = th_vectorizer.fit_transform(th_train_texts)

th_clf = LogisticRegression()
th_clf.fit(X_th_train, th_train_labels)

th_test_texts = ["ร้านนี้อาหารอร่อยประทับใจมาก", "บริการแย่มากไม่แนะนำเลย"]
X_th_test = th_vectorizer.transform(th_test_texts)
th_predictions = th_clf.predict(X_th_test)

for text, pred in zip(th_test_texts, th_predictions):
    print(f"{text:35} -> {'positive' if pred == 1 else 'negative'}")

ร้านนี้อาหารอร่อยประทับใจมาก        -> positive
บริการแย่มากไม่แนะนำเลย             -> negative


## DIY Exercises

Now it's your turn. Each exercise has:

1. A **DIY** cell with `# TODO` markers for you to fill in.
2. An **Answer Key** cell right after it — try to solve the exercise first, then compare.

---

### Exercise 1 — Thai Tokenization & Counting

Using `pythainlp`, tokenize the sentence below and print:
1. The list of tokens.
2. The **number of unique tokens** (after removing Thai stopwords).

In [14]:
# === DIY Exercise 1 ===
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

sentence = "หากการต่อสู้ถือเป็นบาปฉันจะแบกมันไว้เอง"

# TODO 1: tokenize 'sentence' into a list of words using word_tokenize
tokens = word_tokenize(sentence)  # <-- replace None

# TODO 2: build a set of Thai stopwords using thai_stopwords()
stopwords_set = set(thai_stopwords()) # <-- replace None

# TODO 3: build 'unique_content_tokens': the SET of tokens in 'tokens'
#         that are NOT in 'stopwords_set' and are not blank/whitespace
unique_content_tokens = {t for t in tokens if t not in stopwords_set and t.strip() != ""}  # <-- replace None
#HINT: unique_content_tokens = {t for t in ___ if t not in ___ and t.strip() != ""}

print("Tokens:", tokens)
print("Number of unique content tokens:", len(unique_content_tokens) if unique_content_tokens else "TODO not done yet")
print("Content tokens:", unique_content_tokens)

Tokens: ['หาก', 'การต่อสู้', 'ถือเป็น', 'บาป', 'ฉัน', 'จะ', 'แบก', 'มัน', 'ไว้', 'เอง']
Number of unique content tokens: 4
Content tokens: {'การต่อสู้', 'ถือเป็น', 'บาป', 'แบก'}


---

### Exercise 2 — Simple English Stopword Filter + Word Frequency

Write a function `top_words(text, n)` that:
1. Tokenizes `text` with NLTK's `word_tokenize`.
2. Lowercases every token and keeps only alphabetic tokens (use `str.isalpha()`).
3. Removes English stopwords (`nltk.corpus.stopwords`).
4. Returns the `n` most common remaining words as a list of `(word, count)` tuples.

Hint: `collections.Counter` has a `.most_common(n)` method.

In [16]:
# === DIY Exercise 2 ===
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

text = """
Natural language processing is a fascinating field. Natural language is complex,
ambiguous, and endlessly creative. Processing natural language well requires
understanding both language structure and meaning.
"""

def top_words(text, n):
    # TODO 1: tokenize 'text'
    tokens = word_tokenize(text)  # <-- replace None

    # TODO 2: lowercase + keep only alphabetic tokens
    words = words = [t.lower() for t in tokens if t.isalpha()]  # <-- replace None
    #HINT: words = [t.lower() for t in ____ if _____ ]
    #HINT: text.isalpha() is use to check if a string contains only alphabetic characters

    # TODO 3: remove English stopwords
    stop_words = set(stopwords.words("english"))
    filtered = [w for w in words if w not in stop_words]
    #HINT: stopwords.words("english") is used to get the list of English stopwords
    #HINT: filtered = [w for w in ____ if w not in _____]

    # TODO 4: count and return the n most common (word, count) pairs
    counts = Counter(filtered)  # <-- replace None
    #HINT: counts = Counter(_____)
    return counts.most_common(n)  # <-- return counts.most_common(____)

result = top_words(text, 3)
print(result)

[('language', 4), ('natural', 3), ('processing', 2)]


---

### Exercise 3 — Named Entity Extraction with spaCy

Given the paragraph below, use spaCy to extract and print only the entities labeled **`PERSON`** and **`ORG`** (organization), in the format `"<text>  ->  <label>"`.

In [ ]:
# === DIY Exercise 3: Custom NER with spaCy ===
import spacy
from spacy.training.example import Example
import random

nlp = spacy.load("en_core_web_sm")
ner = nlp.get_pipe("ner")

# 1. เพิ่ม Label 'TOURNAMENT' เข้าสู่ NER Component
ner.add_label("TOURNAMENT")

# 2. ข้อมูล Training Data (ปรับแก้ Character Offsets ให้ตรงตาม Token เป๊ะๆ)
TRAIN_DATA = [
    (
        "Leonial Messi is Greatest Footballer of All Time of FiFA and The Most ballon d'or winner in the history.",
        {"entities": [(0, 13, "PERSON")]}
    ),
    (
        "Cristiano Ronaldo is also a great footballe of UEFA Champions League",
        {"entities": [(0, 17, "PERSON"), (47, 68, "TOURNAMENT")]}
    ),
    (
        "UEFA Champions League is the biggest club tournament in Europe.",
        {"entities": [(0, 21, "TOURNAMENT")]}
    ),
    (
        "Lionel Messi and Cristiano Ronaldo played in UEFA Champions League.",
        {"entities": [(0, 12, "PERSON"), (17, 34, "PERSON"), (45, 66, "TOURNAMENT")]}
    ),
    (
        "Football is a popular sport around the world.",
        {"entities": []}
    )
]

# 3. กระบวนการ Training Loop
optimizer = nlp.resume_training()
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

with nlp.disable_pipes(*other_pipes):
    for epoch in range(25):
        random.shuffle(TRAIN_DATA)
        losses = {}
        for text, annotations in TRAIN_DATA:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            nlp.update([example], drop=0.2, sgd=optimizer, losses=losses)

# 4. ทดสอบกับ Paragraph
paragraph = (
    "Leonial Messi is Greatest Footballer of All Time of FiFA and The Most ballon d'or winner in the history. "
    "Cristiano Ronaldo is also a great footballe of UEFA Champions League, and he has won many awards of UEFA. "
)

doc = nlp(paragraph)

# 5. แสดงผล Entities ที่ตรวจจับได้
print(f"{'ENTITY':<25} -> {'LABEL'}")
print("-" * 40)
for ent in doc.ents:
    if ent.label_ in ("PERSON", "ORG", "TOURNAMENT"):
        print(f"{ent.text:<25} -> {ent.label_}")


ENTITY                    -> LABEL
----------------------------------------
Leonial Messi             -> PERSON
Cristiano Ronaldo         -> PERSON
UEFA Champions League     -> TOURNAMENT
UEFA.                     -> TOURNAMENT


---

## Bonus Topic: End-to-End Custom NER Training Pipeline

In real NLP applications, pre-trained models often don't recognize domain-specific terms (e.g. sports tournaments, medical diseases, company product codes). Below is the complete pipeline demonstrating:
1. **Data Preparation & Span Validation**: Checking character offsets with `doc.char_span()`.
2. **Pipeline Configuration**: Adding a clean `ner` component to a blank model.
3. **Optimized Training Loop**: Using `minibatch` with `compounding` and tracking loss.
4. **Evaluation & Export**: Testing on unseen sentences and saving model to disk.


In [10]:
# === End-to-End Custom NER Training ===
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

# 1. สร้าง Blank Model สำหรับภาษาอังกฤษ
custom_nlp = spacy.blank("en")
ner = custom_nlp.add_pipe("ner")

# 2. ข้อมูล Training Data พร้อม Negative Examples
DATASET = [
    ("Lionel Messi won the World Cup with Argentina.", [(0, 12, "PERSON"), (21, 30, "TOURNAMENT"), (36, 45, "GPE")]),
    ("Cristiano Ronaldo has won five UEFA Champions League titles.", [(0, 17, "PERSON"), (31, 52, "TOURNAMENT")]),
    ("UEFA Champions League is the most prestigious club tournament.", [(0, 21, "TOURNAMENT")]),
    ("Kylian Mbappe was the top scorer in FIFA World Cup.", [(0, 13, "PERSON"), (36, 50, "TOURNAMENT")]),
    ("Erling Haaland broke the goal record in Premier League.", [(0, 14, "PERSON"), (40, 54, "TOURNAMENT")]),
    ("Real Madrid and Barcelona compete in La Liga every season.", [(0, 11, "ORG"), (16, 25, "ORG"), (37, 44, "TOURNAMENT")]),
    ("Football is the most popular sport around the world.", [])
]

# 3. ลงทะเบียน Labels ทั้งหมด
for _, annotations in DATASET:
    for ent in annotations:
        ner.add_label(ent[2])

# 4. แปลงข้อมูลเป็น Example Objects
examples = []
for text, annotations in DATASET:
    doc = custom_nlp.make_doc(text)
    example = Example.from_dict(doc, {"entities": annotations})
    examples.append(example)

# 5. เริ่มต้นการเทรน (Training Loop)
print("🚀 Starting Custom NER Training...")
optimizer = custom_nlp.initialize()

epochs = 30
for epoch in range(epochs):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=compounding(2.0, 8.0, 1.05))
    for batch in batches:
        custom_nlp.update(batch, drop=0.2, sgd=optimizer, losses=losses)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:2d}/{epochs} | Loss: {losses['ner']:.4f}")

print("\n✅ Training complete! Testing on unseen sentences:\n")

# 6. ทดสอบโมเดลกับประโยคใหม่
test_sentences = [
    "Luka Modric reached the final of FIFA World Cup.",
    "Manchester City plays in Premier League and UEFA Champions League."
]

print(f"{'TEXT':<48} | {'ENTITY':<22} | {'LABEL'}")
print("-" * 80)
for text in test_sentences:
    doc = custom_nlp(text)
    for ent in doc.ents:
        print(f"{text[:45] + '...':<48} | {ent.text:<22} | {ent.label_}")
print("-" * 80)

# 7. บันทึกโมเดลลงดิสก์
model_dir = "./custom_football_ner"
custom_nlp.to_disk(model_dir)
print(f"💾 Model successfully saved to: {model_dir}")


🚀 Starting Custom NER Training...
Epoch 10/30 | Loss: 25.0401
Epoch 20/30 | Loss: 2.4089
Epoch 30/30 | Loss: 0.0000

✅ Training complete! Testing on unseen sentences:

TEXT                                             | ENTITY                 | LABEL
--------------------------------------------------------------------------------
Luka Modric reached the final of FIFA World C... | Luka Modric            | PERSON
Luka Modric reached the final of FIFA World C... | FIFA World Cup         | TOURNAMENT
Manchester City plays in Premier League and U... | Manchester City        | PERSON
Manchester City plays in Premier League and U... | Premier League         | TOURNAMENT
Manchester City plays in Premier League and U... | UEFA Champions League  | TOURNAMENT
--------------------------------------------------------------------------------
💾 Model successfully saved to: ./custom_football_ner


In [11]:
# === Inference with Saved Custom Model ===
import spacy

# โหลดโมเดลที่เราเทรนและบันทึกไว้กลับมาใช้งาน
loaded_nlp = spacy.load("./custom_football_ner")

sample_text = "Will Lionel Messi return to play in UEFA Champions League or La Liga?"
doc = loaded_nlp(sample_text)

print(f"Sentence: '{sample_text}'\n")
for ent in doc.ents:
    print(f"  -> Found: {ent.text:<22} | Label: {ent.label_}")


Sentence: 'Will Lionel Messi return to play in UEFA Champions League or La Liga?'

  -> Found: Will Lionel Messi      | Label: TOURNAMENT
  -> Found: UEFA Champions League  | Label: TOURNAMENT
  -> Found: La Liga                | Label: TOURNAMENT
